In [ ]:
%pip install xgboost

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn

from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True

except ImportError:
    XGBOOST_AVAILABLE = False

print("Scikit-learn version:", sklearn.__version__)
print("XGBoost available:", XGBOOST_AVAILABLE)

if not XGBOOST_AVAILABLE:
    raise ImportError(
        "XGBoost is not installed. Run the installation cell "
        "and then restart the Jupyter kernel."
    )

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

ML_FIGURE_DIR = (
    PROJECT_ROOT
    / "reports"
    / "figures"
    / "machine_learning"
)

ML_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FACTOR_SCORE_FILE = (
    PROCESSED_DATA_DIR
    / "11_sp500_monthly_factor_scores_full_2005_2025.parquet"
)

FF_FACTOR_FILE = (
    PROCESSED_DATA_DIR
    / "15_fama_french_monthly_2015_2025.parquet"
)

BENCHMARK_FILE = (
    PROCESSED_DATA_DIR
    / "12_benchmark_returns_monthly_2005_2025.parquet"
)

ml_factor_df = pd.read_parquet(
    FACTOR_SCORE_FILE
)

ml_factor_df["month"] = pd.to_datetime(
    ml_factor_df["month"]
)

ml_factor_df["return_month"] = (
    ml_factor_df["month"]
    + pd.offsets.MonthEnd(1)
)

ml_factor_df = (
    ml_factor_df
    .sort_values(
        ["month", "permno"]
    )
    .reset_index(drop=True)
)

print("Machine-learning factor data was loaded successfully.")
print("Number of rows:", len(ml_factor_df))
print("Number of columns:", len(ml_factor_df.columns))
print(
    "Date range:",
    ml_factor_df["month"].min().date(),
    "to",
    ml_factor_df["month"].max().date()
)

In [ ]:
ml_feature_columns = [
    "factor_value",
    "factor_momentum",
    "factor_quality",
    "factor_investment",
    "factor_size",
    "factor_low_volatility"
]

ml_required_columns = [
    "permno",
    "ticker",
    "month",
    "return_month",
    "future_return_1m",
    "month_end_market_cap",
    "volatility_12m",
    "multi_factor_score"
] + ml_feature_columns

missing_ml_columns = [
    column
    for column in ml_required_columns
    if column not in ml_factor_df.columns
]

if missing_ml_columns:
    raise ValueError(
        f"Missing machine-learning columns: {missing_ml_columns}"
    )


def winsorize_cross_section(
    group,
    lower_quantile=0.01,
    upper_quantile=0.99
):
    values = group[
        "future_return_1m"
    ]

    valid_values = values.dropna()

    if len(valid_values) < 20:
        group[
            "future_return_1m_winsorized"
        ] = values

        return group

    lower_bound = valid_values.quantile(
        lower_quantile
    )

    upper_bound = valid_values.quantile(
        upper_quantile
    )

    group[
        "future_return_1m_winsorized"
    ] = values.clip(
        lower=lower_bound,
        upper=upper_bound
    )

    return group


ml_model_df = (
    ml_factor_df
    .groupby(
        "month",
        group_keys=False
    )
    .apply(
        winsorize_cross_section
    )
    .reset_index(drop=True)
)

print("Cross-sectional return winsorization was completed.")
print(
    "Available raw targets:",
    ml_model_df[
        "future_return_1m"
    ].notna().sum()
)
print(
    "Available winsorized targets:",
    ml_model_df[
        "future_return_1m_winsorized"
    ].notna().sum()
)

In [ ]:
prediction_years = list(
    range(2015, 2026)
)

walk_forward_plan_records = []

for prediction_year in prediction_years:
    first_formation_month = pd.Timestamp(
        year=prediction_year - 1,
        month=12,
        day=31
    )

    last_formation_month = pd.Timestamp(
        year=prediction_year,
        month=11,
        day=30
    )

    training_sample = (
        ml_model_df
        .loc[
            (
                ml_model_df["month"]
                < first_formation_month
            )
            & (
                ml_model_df[
                    "future_return_1m_winsorized"
                ].notna()
            )
        ]
    )

    prediction_sample = (
        ml_model_df
        .loc[
            (
                ml_model_df["month"]
                >= first_formation_month
            )
            & (
                ml_model_df["month"]
                <= last_formation_month
            )
        ]
    )

    walk_forward_plan_records.append({
        "prediction_year":
            prediction_year,

        "training_cutoff":
            first_formation_month,

        "training_start_month":
            training_sample["month"].min(),

        "training_end_month":
            training_sample["month"].max(),

        "number_of_training_months":
            training_sample["month"].nunique(),

        "number_of_training_rows":
            len(training_sample),

        "prediction_start_month":
            prediction_sample["month"].min(),

        "prediction_end_month":
            prediction_sample["month"].max(),

        "number_of_prediction_months":
            prediction_sample["month"].nunique(),

        "number_of_prediction_rows":
            len(prediction_sample)
    })

walk_forward_plan_df = pd.DataFrame(
    walk_forward_plan_records
)

if (
    walk_forward_plan_df[
        "number_of_prediction_months"
    ]
    != 12
).any():
    raise ValueError(
        "Each prediction year should contain 12 formation months."
    )

print("Annual walk-forward plan:")
display(walk_forward_plan_df)

In [ ]:
ridge_model_parameters = {
    "alpha": 10.0
}

xgboost_model_parameters = {
    "n_estimators": 150,
    "max_depth": 3,
    "learning_rate": 0.03,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "min_child_weight": 20,
    "reg_alpha": 0.10,
    "reg_lambda": 10.0,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0
}


def create_ridge_pipeline():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                Ridge(
                    alpha=ridge_model_parameters[
                        "alpha"
                    ]
                )
            )
        ]
    )


def create_xgboost_pipeline():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "model",
                XGBRegressor(
                    **xgboost_model_parameters
                )
            )
        ]
    )


print("Machine-learning model specifications were created.")
print("Ridge parameters:", ridge_model_parameters)
print("XGBoost parameters:", xgboost_model_parameters)

In [ ]:
ml_prediction_frames = []
ridge_coefficient_records = []
xgboost_importance_records = []
annual_model_diagnostic_records = []

for prediction_year in prediction_years:
    first_formation_month = pd.Timestamp(
        year=prediction_year - 1,
        month=12,
        day=31
    )

    last_formation_month = pd.Timestamp(
        year=prediction_year,
        month=11,
        day=30
    )

    training_sample = (
        ml_model_df
        .loc[
            (
                ml_model_df["month"]
                < first_formation_month
            )
            & (
                ml_model_df[
                    "future_return_1m_winsorized"
                ].notna()
            )
        ]
        .copy()
    )

    prediction_sample = (
        ml_model_df
        .loc[
            (
                ml_model_df["month"]
                >= first_formation_month
            )
            & (
                ml_model_df["month"]
                <= last_formation_month
            )
        ]
        .copy()
    )

    training_features = (
        training_sample[
            ml_feature_columns
        ]
    )

    training_target = (
        training_sample[
            "future_return_1m_winsorized"
        ]
        .astype(float)
    )

    prediction_features = (
        prediction_sample[
            ml_feature_columns
        ]
    )

    ridge_pipeline = (
        create_ridge_pipeline()
    )

    xgboost_pipeline = (
        create_xgboost_pipeline()
    )

    print(
        f"Training models for prediction year "
        f"{prediction_year}..."
    )

    ridge_pipeline.fit(
        training_features,
        training_target
    )

    xgboost_pipeline.fit(
        training_features,
        training_target
    )

    ridge_predictions = (
        ridge_pipeline.predict(
            prediction_features
        )
    )

    xgboost_predictions = (
        xgboost_pipeline.predict(
            prediction_features
        )
    )

    prediction_output = (
        prediction_sample[
            [
                "permno",
                "ticker",
                "month",
                "return_month",
                "future_return_1m",
                "month_end_market_cap",
                "volatility_12m",
                "multi_factor_score"
            ]
            + ml_feature_columns
        ]
        .copy()
    )

    prediction_output[
        "prediction_year"
    ] = prediction_year

    prediction_output[
        "training_cutoff"
    ] = first_formation_month

    prediction_output[
        "ridge_predicted_return"
    ] = ridge_predictions

    prediction_output[
        "xgboost_predicted_return"
    ] = xgboost_predictions

    ml_prediction_frames.append(
        prediction_output
    )

    ridge_coefficients = (
        ridge_pipeline
        .named_steps["model"]
        .coef_
    )

    for feature_name, coefficient in zip(
        ml_feature_columns,
        ridge_coefficients
    ):
        ridge_coefficient_records.append({
            "prediction_year":
                prediction_year,

            "feature":
                feature_name,

            "standardized_coefficient":
                coefficient
        })

    xgboost_importances = (
        xgboost_pipeline
        .named_steps["model"]
        .feature_importances_
    )

    for feature_name, importance in zip(
        ml_feature_columns,
        xgboost_importances
    ):
        xgboost_importance_records.append({
            "prediction_year":
                prediction_year,

            "feature":
                feature_name,

            "feature_importance":
                importance
        })

    ridge_training_prediction = (
        ridge_pipeline.predict(
            training_features
        )
    )

    xgboost_training_prediction = (
        xgboost_pipeline.predict(
            training_features
        )
    )

    ridge_training_rmse = np.sqrt(
        np.mean(
            (
                training_target.to_numpy()
                - ridge_training_prediction
            )
            ** 2
        )
    )

    xgboost_training_rmse = np.sqrt(
        np.mean(
            (
                training_target.to_numpy()
                - xgboost_training_prediction
            )
            ** 2
        )
    )

    annual_model_diagnostic_records.append({
        "prediction_year":
            prediction_year,

        "number_of_training_rows":
            len(training_sample),

        "number_of_prediction_rows":
            len(prediction_sample),

        "ridge_training_rmse":
            ridge_training_rmse,

        "xgboost_training_rmse":
            xgboost_training_rmse
    })

    print(
        f"Prediction year {prediction_year} completed."
    )


ml_prediction_df = pd.concat(
    ml_prediction_frames,
    ignore_index=True
)

ridge_coefficient_df = pd.DataFrame(
    ridge_coefficient_records
)

xgboost_importance_df = pd.DataFrame(
    xgboost_importance_records
)

annual_model_diagnostic_df = pd.DataFrame(
    annual_model_diagnostic_records
)

print("All walk-forward models were trained successfully.")

In [ ]:
duplicate_prediction_count = (
    ml_prediction_df
    .duplicated(
        subset=[
            "permno",
            "month"
        ]
    )
    .sum()
)

look_ahead_violation_count = (
    ml_prediction_df[
        "month"
    ]
    .lt(
        ml_prediction_df[
            "training_cutoff"
        ]
    )
    .sum()
)

missing_ridge_predictions = (
    ml_prediction_df[
        "ridge_predicted_return"
    ]
    .isna()
    .sum()
)

missing_xgboost_predictions = (
    ml_prediction_df[
        "xgboost_predicted_return"
    ]
    .isna()
    .sum()
)

if duplicate_prediction_count != 0:
    raise ValueError(
        "Duplicate PERMNO-month predictions were found."
    )

if look_ahead_violation_count != 0:
    raise ValueError(
        "A look-ahead violation was detected."
    )

if (
    missing_ridge_predictions != 0
    or missing_xgboost_predictions != 0
):
    raise ValueError(
        "Missing machine-learning predictions were found."
    )

prediction_validation_df = pd.DataFrame({
    "total_prediction_rows": [
        len(ml_prediction_df)
    ],

    "unique_prediction_months": [
        ml_prediction_df[
            "month"
        ].nunique()
    ],

    "duplicate_predictions": [
        duplicate_prediction_count
    ],

    "look_ahead_violations": [
        look_ahead_violation_count
    ],

    "missing_ridge_predictions": [
        missing_ridge_predictions
    ],

    "missing_xgboost_predictions": [
        missing_xgboost_predictions
    ],

    "missing_realized_returns": [
        ml_prediction_df[
            "future_return_1m"
        ].isna().sum()
    ]
})

print("Out-of-sample prediction validation:")
display(prediction_validation_df)

print("\nAnnual model diagnostics:")
display(
    annual_model_diagnostic_df.round(6)
)

In [ ]:
duplicate_column_names = (
    ml_prediction_df.columns[
        ml_prediction_df.columns.duplicated()
    ]
    .tolist()
)

print(
    "Duplicate prediction columns before cleaning:",
    duplicate_column_names
)

ml_prediction_df = (
    ml_prediction_df
    .loc[
        :,
        ~ml_prediction_df.columns.duplicated()
    ]
    .copy()
)

print(
    "Duplicate prediction columns after cleaning:",
    ml_prediction_df.columns.duplicated().sum()
)

In [ ]:
import statsmodels.api as sm

from scipy.stats import (
    pearsonr,
    spearmanr
)

prediction_signal_mapping = {
    "Quality Factor":
        "factor_quality",

    "Traditional Six-Factor Score":
        "multi_factor_score",

    "Ridge Prediction":
        "ridge_predicted_return",

    "XGBoost Prediction":
        "xgboost_predicted_return"
}


def calculate_monthly_prediction_ic(
    data,
    signal_name,
    signal_column,
    correlation_method
):
    monthly_records = []

    for month, month_sample in (
        data
        .groupby(
            "month",
            sort=True
        )
    ):
        valid_sample = (
            month_sample[
                [
                    signal_column,
                    "future_return_1m"
                ]
            ]
            .dropna()
            .copy()
        )

        if len(valid_sample) < 30:
            continue

        signal_values = (
            pd.to_numeric(
                valid_sample[
                    signal_column
                ],
                errors="coerce"
            )
            .to_numpy(dtype=float)
        )

        future_returns = (
            pd.to_numeric(
                valid_sample[
                    "future_return_1m"
                ],
                errors="coerce"
            )
            .to_numpy(dtype=float)
        )

        finite_mask = (
            np.isfinite(signal_values)
            & np.isfinite(future_returns)
        )

        signal_values = signal_values[
            finite_mask
        ]

        future_returns = future_returns[
            finite_mask
        ]

        if len(signal_values) < 30:
            continue

        if correlation_method == "spearman":
            information_coefficient = (
                spearmanr(
                    signal_values,
                    future_returns
                )[0]
            )

        elif correlation_method == "pearson":
            information_coefficient = (
                pearsonr(
                    signal_values,
                    future_returns
                )[0]
            )

        else:
            raise ValueError(
                "Unknown correlation method."
            )

        monthly_records.append({
            "month":
                month,

            "signal":
                signal_name,

            "correlation_method":
                correlation_method,

            "number_of_stocks":
                len(signal_values),

            "information_coefficient":
                information_coefficient
        })

    return monthly_records


monthly_prediction_ic_records = []

for signal_name, signal_column in (
    prediction_signal_mapping.items()
):
    for correlation_method in [
        "spearman",
        "pearson"
    ]:
        monthly_prediction_ic_records.extend(
            calculate_monthly_prediction_ic(
                data=ml_prediction_df,
                signal_name=signal_name,
                signal_column=signal_column,
                correlation_method=correlation_method
            )
        )

monthly_prediction_ic_df = pd.DataFrame(
    monthly_prediction_ic_records
)


def summarize_ic_with_hac(group):
    ic_series = (
        group[
            "information_coefficient"
        ]
        .dropna()
        .astype(float)
    )

    constant = np.ones(
        shape=(
            len(ic_series),
            1
        )
    )

    model = sm.OLS(
        ic_series.to_numpy(),
        constant
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3
        }
    )

    monthly_standard_deviation = (
        ic_series.std(ddof=1)
    )

    annualized_icir = (
        ic_series.mean()
        / monthly_standard_deviation
        * np.sqrt(12.0)
        if monthly_standard_deviation > 0
        else np.nan
    )

    return pd.Series({
        "number_of_months":
            len(ic_series),

        "mean_ic":
            ic_series.mean(),

        "annualized_icir":
            annualized_icir,

        "newey_west_t_statistic":
            model.tvalues[0],

        "newey_west_p_value":
            model.pvalues[0],

        "positive_month_rate":
            (ic_series > 0).mean()
    })


prediction_ic_summary_df = (
    monthly_prediction_ic_df
    .groupby(
        [
            "signal",
            "correlation_method"
        ]
    )
    .apply(
        summarize_ic_with_hac
    )
    .reset_index()
)

print("Out-of-sample prediction IC summary:")
display(
    prediction_ic_summary_df.round(4)
)

In [ ]:
ridge_coefficient_pivot_df = (
    ridge_coefficient_df
    .pivot(
        index="prediction_year",
        columns="feature",
        values="standardized_coefficient"
    )
)

xgboost_importance_pivot_df = (
    xgboost_importance_df
    .pivot(
        index="prediction_year",
        columns="feature",
        values="feature_importance"
    )
)

average_feature_summary_df = pd.DataFrame({
    "ridge_average_coefficient":
        ridge_coefficient_df
        .groupby("feature")[
            "standardized_coefficient"
        ]
        .mean(),

    "ridge_coefficient_standard_deviation":
        ridge_coefficient_df
        .groupby("feature")[
            "standardized_coefficient"
        ]
        .std(ddof=1),

    "ridge_positive_year_rate":
        ridge_coefficient_df
        .groupby("feature")[
            "standardized_coefficient"
        ]
        .apply(
            lambda values: (
                values > 0
            ).mean()
        ),

    "xgboost_average_importance":
        xgboost_importance_df
        .groupby("feature")[
            "feature_importance"
        ]
        .mean(),

    "xgboost_importance_standard_deviation":
        xgboost_importance_df
        .groupby("feature")[
            "feature_importance"
        ]
        .std(ddof=1)
})

print("Average machine-learning feature summary:")
display(
    average_feature_summary_df
    .sort_values(
        "xgboost_average_importance",
        ascending=False
    )
    .round(6)
)

In [ ]:
ml_portfolio_signal_mapping = {
    "Quality Factor Top 20%":
        "factor_quality",

    "Traditional Six-Factor Top 20%":
        "multi_factor_score",

    "Ridge Prediction Top 20%":
        "ridge_predicted_return",

    "XGBoost Prediction Top 20%":
        "xgboost_predicted_return"
}

ml_selection_fraction = 0.20

ml_target_weight_frames = []

for month, month_sample in (
    ml_prediction_df
    .groupby(
        "month",
        sort=True
    )
):
    for strategy_name, signal_column in (
        ml_portfolio_signal_mapping.items()
    ):
        eligible_sample = (
            month_sample
            .loc[
                month_sample[
                    signal_column
                ].notna()
            ]
            .copy()
        )

        eligible_sample[
            "signal_percentile_rank"
        ] = (
            eligible_sample[
                signal_column
            ]
            .rank(
                method="first",
                pct=True
            )
        )

        selected_sample = (
            eligible_sample
            .loc[
                eligible_sample[
                    "signal_percentile_rank"
                ]
                > (
                    1.0
                    - ml_selection_fraction
                )
            ]
            .copy()
        )

        if len(selected_sample) == 0:
            raise ValueError(
                f"No holdings were selected for "
                f"{strategy_name} in {month}."
            )

        selected_sample[
            "target_weight"
        ] = (
            1.0
            / len(selected_sample)
        )

        selected_sample["strategy"] = (
            strategy_name
        )

        selected_sample["signal_column"] = (
            signal_column
        )

        ml_target_weight_frames.append(
            selected_sample[
                [
                    "strategy",
                    "signal_column",
                    "month",
                    "return_month",
                    "permno",
                    "ticker",
                    "target_weight",
                    "future_return_1m"
                ]
            ].copy()
        )

ml_target_weights_df = pd.concat(
    ml_target_weight_frames,
    ignore_index=True
)

print("Machine-learning target portfolios were created successfully.")
print(
    "Number of target-weight records:",
    len(ml_target_weights_df)
)
print(
    "Number of strategies:",
    ml_target_weights_df[
        "strategy"
    ].nunique()
)

In [ ]:
ml_weight_validation_df = (
    ml_target_weights_df
    .groupby(
        [
            "strategy",
            "month"
        ],
        as_index=False
    )
    .agg(
        weight_sum=(
            "target_weight",
            "sum"
        ),

        number_of_holdings=(
            "permno",
            "nunique"
        ),

        missing_future_returns=(
            "future_return_1m",
            lambda values: values.isna().sum()
        )
    )
)

maximum_ml_weight_error = (
    ml_weight_validation_df[
        "weight_sum"
    ]
    .sub(1.0)
    .abs()
    .max()
)

duplicate_ml_holdings = (
    ml_target_weights_df
    .duplicated(
        subset=[
            "strategy",
            "month",
            "permno"
        ]
    )
    .sum()
)

if maximum_ml_weight_error > 1e-10:
    raise ValueError(
        "Machine-learning target weights do not sum to one."
    )

if duplicate_ml_holdings != 0:
    raise ValueError(
        "Duplicate machine-learning holdings were found."
    )

ml_holdings_summary_df = (
    ml_weight_validation_df
    .groupby("strategy")
    .agg(
        minimum_holdings=(
            "number_of_holdings",
            "min"
        ),

        average_holdings=(
            "number_of_holdings",
            "mean"
        ),

        maximum_holdings=(
            "number_of_holdings",
            "max"
        ),

        total_missing_future_returns=(
            "missing_future_returns",
            "sum"
        )
    )
)

print("Machine-learning target-weight validation:")
print(
    "Maximum absolute weight error:",
    f"{maximum_ml_weight_error:.12f}"
)
print(
    "Duplicate holdings:",
    duplicate_ml_holdings
)

display(
    ml_holdings_summary_df.round(2)
)

In [ ]:
ml_transaction_cost_rate = 0.001


def run_ml_portfolio_backtest(
    target_weight_data,
    transaction_cost_rate
):
    monthly_records = []

    for strategy_name, strategy_sample in (
        target_weight_data
        .groupby(
            "strategy",
            sort=True
        )
    ):
        strategy_sample = (
            strategy_sample
            .sort_values(
                [
                    "month",
                    "permno"
                ]
            )
        )

        previous_drifted_weights = None

        for month, month_sample in (
            strategy_sample
            .groupby(
                "month",
                sort=True
            )
        ):
            target_weights = (
                month_sample
                .set_index("permno")[
                    "target_weight"
                ]
                .astype(float)
            )

            realized_returns = (
                month_sample
                .set_index("permno")[
                    "future_return_1m"
                ]
                .astype(float)
            )

            missing_return_count = int(
                realized_returns
                .isna()
                .sum()
            )

            realized_returns_filled = (
                realized_returns
                .fillna(0.0)
            )

            if previous_drifted_weights is None:
                turnover = 1.0

            else:
                combined_permnos = (
                    target_weights.index
                    .union(
                        previous_drifted_weights.index
                    )
                )

                current_weights_aligned = (
                    target_weights
                    .reindex(
                        combined_permnos,
                        fill_value=0.0
                    )
                )

                previous_weights_aligned = (
                    previous_drifted_weights
                    .reindex(
                        combined_permnos,
                        fill_value=0.0
                    )
                )

                turnover = (
                    0.5
                    * (
                        current_weights_aligned
                        - previous_weights_aligned
                    )
                    .abs()
                    .sum()
                )

            gross_return = (
                target_weights
                * realized_returns_filled
            ).sum()

            transaction_cost = (
                turnover
                * transaction_cost_rate
            )

            net_return = (
                (
                    1.0
                    - transaction_cost
                )
                * (
                    1.0
                    + gross_return
                )
                - 1.0
            )

            end_of_month_values = (
                target_weights
                * (
                    1.0
                    + realized_returns_filled
                )
            )

            total_end_of_month_value = (
                end_of_month_values.sum()
            )

            previous_drifted_weights = (
                end_of_month_values
                / total_end_of_month_value
            )

            monthly_records.append({
                "strategy":
                    strategy_name,

                "formation_month":
                    month,

                "return_month":
                    month_sample[
                        "return_month"
                    ].iloc[0],

                "number_of_holdings":
                    len(target_weights),

                "missing_return_holdings":
                    missing_return_count,

                "turnover":
                    turnover,

                "transaction_cost":
                    transaction_cost,

                "gross_return":
                    gross_return,

                "net_return":
                    net_return
            })

    return pd.DataFrame(
        monthly_records
    )


ml_strategy_backtest_df = (
    run_ml_portfolio_backtest(
        target_weight_data=ml_target_weights_df,
        transaction_cost_rate=ml_transaction_cost_rate
    )
)

print("Machine-learning portfolio backtests were completed.")
print(
    "Number of monthly records:",
    len(ml_strategy_backtest_df)
)
print(
    "Number of strategies:",
    ml_strategy_backtest_df[
        "strategy"
    ].nunique()
)

print("\nMonths per strategy:")
print(
    ml_strategy_backtest_df
    .groupby("strategy")
    .size()
)

In [ ]:
ml_ff_df = pd.read_parquet(
    FF_FACTOR_FILE
)

ml_benchmark_df = pd.read_parquet(
    BENCHMARK_FILE
)

ml_ff_df["month"] = pd.to_datetime(
    ml_ff_df["month"]
)

ml_benchmark_df["month"] = pd.to_datetime(
    ml_benchmark_df["month"]
)

ml_strategy_return_long_df = (
    ml_strategy_backtest_df[
        [
            "return_month",
            "strategy",
            "net_return",
            "turnover",
            "number_of_holdings",
            "missing_return_holdings"
        ]
    ]
    .copy()
)

benchmark_long_frames = []

benchmark_specifications = {
    "CRSP S&P 500 Value Weighted":
        "crsp_value_weighted_total_return",

    "CRSP S&P 500 Equal Weighted":
        "crsp_equal_weighted_total_return"
}

for benchmark_name, return_column in (
    benchmark_specifications.items()
):
    benchmark_sample = pd.DataFrame({
        "return_month":
            ml_benchmark_df["month"],

        "strategy":
            benchmark_name,

        "net_return":
            ml_benchmark_df[return_column],

        "turnover":
            np.nan,

        "number_of_holdings":
            np.nan,

        "missing_return_holdings":
            0
    })

    benchmark_long_frames.append(
        benchmark_sample
    )

ml_comparison_long_df = pd.concat(
    [
        ml_strategy_return_long_df
    ]
    + benchmark_long_frames,
    ignore_index=True
)

ml_comparison_long_df = (
    ml_comparison_long_df
    .merge(
        ml_ff_df.rename(
            columns={
                "month":
                    "return_month"
            }
        ),
        on="return_month",
        how="inner",
        validate="many_to_one"
    )
    .merge(
        ml_benchmark_df[
            [
                "month",
                "crsp_equal_weighted_total_return"
            ]
        ].rename(
            columns={
                "month":
                    "return_month"
            }
        ),
        on="return_month",
        how="left",
        validate="many_to_one"
    )
    .sort_values(
        [
            "strategy",
            "return_month"
        ]
    )
    .reset_index(drop=True)
)

print("Machine-learning returns and benchmark data were merged.")
print(
    "Number of comparison series:",
    ml_comparison_long_df[
        "strategy"
    ].nunique()
)

In [ ]:
def calculate_ml_portfolio_performance(
    group
):
    group = (
        group
        .sort_values("return_month")
        .copy()
    )

    returns = group[
        "net_return"
    ].astype(float)

    excess_returns = (
        returns
        - group["rf"].astype(float)
    )

    equal_weighted_benchmark = (
        group[
            "crsp_equal_weighted_total_return"
        ]
        .astype(float)
    )

    active_returns = (
        returns
        - equal_weighted_benchmark
    )

    number_of_months = len(returns)

    terminal_wealth = (
        1.0 + returns
    ).prod()

    annualized_return = (
        terminal_wealth
        ** (
            12.0
            / number_of_months
        )
        - 1.0
    )

    annualized_volatility = (
        returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    annualized_sharpe = (
        excess_returns.mean()
        / excess_returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    downside_returns = np.minimum(
        excess_returns,
        0.0
    )

    downside_deviation = (
        np.sqrt(
            np.mean(
                downside_returns ** 2
            )
        )
        * np.sqrt(12.0)
    )

    annualized_sortino = (
        excess_returns.mean()
        * 12.0
        / downside_deviation
        if downside_deviation > 0
        else np.nan
    )

    wealth = (
        1.0 + returns
    ).cumprod()

    drawdown = (
        wealth
        / wealth.cummax()
        - 1.0
    )

    fifth_percentile = (
        returns.quantile(0.05)
    )

    historical_cvar_95 = (
        -returns[
            returns <= fifth_percentile
        ].mean()
    )

    tracking_error = (
        active_returns.std(ddof=1)
        * np.sqrt(12.0)
    )

    information_ratio = (
        active_returns.mean()
        * 12.0
        / tracking_error
        if tracking_error > 0
        else np.nan
    )

    return pd.Series({
        "number_of_months":
            number_of_months,

        "annualized_return":
            annualized_return,

        "annualized_volatility":
            annualized_volatility,

        "annualized_sharpe":
            annualized_sharpe,

        "annualized_sortino":
            annualized_sortino,

        "maximum_drawdown":
            drawdown.min(),

        "historical_cvar_95":
            historical_cvar_95,

        "average_turnover":
            group["turnover"].mean(),

        "annualized_active_return":
            active_returns.mean() * 12.0,

        "annualized_tracking_error":
            tracking_error,

        "information_ratio":
            information_ratio,

        "terminal_wealth":
            terminal_wealth
    })


ml_performance_summary_df = (
    ml_comparison_long_df
    .groupby(
        "strategy",
        sort=True
    )
    .apply(
        calculate_ml_portfolio_performance
    )
    .sort_values(
        "annualized_sharpe",
        ascending=False
    )
)

print("Machine-learning portfolio performance:")
display(
    ml_performance_summary_df.round(4)
)

In [ ]:
ml_factor_columns = [
    "mkt_rf",
    "smb",
    "hml",
    "rmw",
    "cma",
    "mom"
]

ml_alpha_records = []

for strategy_name, group in (
    ml_comparison_long_df
    .groupby(
        "strategy",
        sort=True
    )
):
    regression_sample = (
        group
        .dropna(
            subset=[
                "net_return",
                "rf"
            ]
            + ml_factor_columns
        )
    )

    excess_return = (
        regression_sample["net_return"]
        - regression_sample["rf"]
    )

    independent_variables = sm.add_constant(
        regression_sample[
            ml_factor_columns
        ],
        has_constant="add"
    )

    model = sm.OLS(
        excess_return,
        independent_variables
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3
        }
    )

    ml_alpha_records.append({
        "strategy":
            strategy_name,

        "annualized_alpha":
            model.params["const"] * 12.0,

        "alpha_t_statistic":
            model.tvalues["const"],

        "alpha_p_value":
            model.pvalues["const"],

        "mkt_rf_beta":
            model.params["mkt_rf"],

        "smb_beta":
            model.params["smb"],

        "hml_beta":
            model.params["hml"],

        "rmw_beta":
            model.params["rmw"],

        "cma_beta":
            model.params["cma"],

        "mom_beta":
            model.params["mom"],

        "adjusted_r_squared":
            model.rsquared_adj
    })

ml_alpha_summary_df = (
    pd.DataFrame(
        ml_alpha_records
    )
    .set_index("strategy")
    .sort_values(
        "annualized_alpha",
        ascending=False
    )
)

print("Machine-learning factor-adjusted performance:")
display(
    ml_alpha_summary_df.round(4)
)

In [ ]:
ml_return_wide_df = (
    ml_comparison_long_df
    .pivot(
        index="return_month",
        columns="strategy",
        values="net_return"
    )
    .sort_index()
)

ml_plot_order = [
    "Quality Factor Top 20%",
    "Traditional Six-Factor Top 20%",
    "Ridge Prediction Top 20%",
    "XGBoost Prediction Top 20%",
    "CRSP S&P 500 Value Weighted",
    "CRSP S&P 500 Equal Weighted"
]

ml_cumulative_wealth_df = (
    1.0
    + ml_return_wide_df[
        ml_plot_order
    ]
).cumprod()

ml_color_mapping = {
    "Quality Factor Top 20%":
        "#1f77b4",

    "Traditional Six-Factor Top 20%":
        "#d62728",

    "Ridge Prediction Top 20%":
        "#9467bd",

    "XGBoost Prediction Top 20%":
        "#ff7f0e",

    "CRSP S&P 500 Value Weighted":
        "#2ca02c",

    "CRSP S&P 500 Equal Weighted":
        "#7f7f7f"
}

fig, axis = plt.subplots(
    figsize=(12, 7)
)

plot_dates = (
    ml_cumulative_wealth_df
    .index
    .to_numpy()
)

for series_name in ml_plot_order:
    axis.plot(
        plot_dates,
        ml_cumulative_wealth_df[
            series_name
        ].to_numpy(dtype=float),
        label=series_name,
        color=ml_color_mapping[
            series_name
        ],
        linewidth=2.0
    )

axis.set_title(
    "Out-of-Sample Machine-Learning Portfolio Performance",
    fontsize=15,
    pad=15
)

axis.set_xlabel("Date")
axis.set_ylabel("Growth of $1 Investment")

axis.legend(
    frameon=False,
    loc="upper left",
    fontsize=9
)

axis.grid(alpha=0.25)

fig.tight_layout()

ML_CUMULATIVE_FIGURE = (
    ML_FIGURE_DIR
    / "01_ml_cumulative_wealth_2015_2025.png"
)

fig.savefig(
    ML_CUMULATIVE_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Machine-learning cumulative-wealth figure was saved.")

In [ ]:
rank_ic_wide_df = (
    monthly_prediction_ic_df
    .loc[
        monthly_prediction_ic_df[
            "correlation_method"
        ]
        == "spearman"
    ]
    .pivot(
        index="month",
        columns="signal",
        values="information_coefficient"
    )
    .sort_index()
)

cumulative_rank_ic_df = (
    rank_ic_wide_df.cumsum()
)

ic_color_mapping = {
    "Quality Factor":
        "#1f77b4",

    "Traditional Six-Factor Score":
        "#d62728",

    "Ridge Prediction":
        "#9467bd",

    "XGBoost Prediction":
        "#ff7f0e"
}

fig, axis = plt.subplots(
    figsize=(12, 7)
)

plot_dates = (
    cumulative_rank_ic_df
    .index
    .to_numpy()
)

for signal_name in cumulative_rank_ic_df.columns:
    axis.plot(
        plot_dates,
        cumulative_rank_ic_df[
            signal_name
        ].to_numpy(dtype=float),
        label=signal_name,
        color=ic_color_mapping[
            signal_name
        ],
        linewidth=2.0
    )

axis.axhline(
    y=0,
    color="black",
    linewidth=0.8,
    linestyle="--"
)

axis.set_title(
    "Cumulative Out-of-Sample Rank IC",
    fontsize=15,
    pad=15
)

axis.set_xlabel("Formation Month")
axis.set_ylabel("Cumulative Rank IC")

axis.legend(
    frameon=False,
    loc="best"
)

axis.grid(alpha=0.25)

fig.tight_layout()

CUMULATIVE_IC_FIGURE = (
    ML_FIGURE_DIR
    / "02_cumulative_out_of_sample_rank_ic.png"
)

fig.savefig(
    CUMULATIVE_IC_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Cumulative out-of-sample IC figure was saved.")

In [ ]:
feature_plot_df = (
    average_feature_summary_df
    .copy()
    .sort_values(
        "xgboost_average_importance",
        ascending=True
    )
)

feature_label_mapping = {
    "factor_value":
        "Value",

    "factor_momentum":
        "Momentum",

    "factor_quality":
        "Quality",

    "factor_investment":
        "Investment",

    "factor_size":
        "Size",

    "factor_low_volatility":
        "Low Volatility"
}

feature_plot_labels = [
    feature_label_mapping[
        feature
    ]
    for feature in feature_plot_df.index
]

figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13, 6)
)

axes[0].barh(
    feature_plot_labels,
    feature_plot_df[
        "ridge_average_coefficient"
    ].to_numpy(dtype=float),
    color="#9467bd"
)

axes[0].axvline(
    x=0,
    color="black",
    linewidth=0.8
)

axes[0].set_title(
    "Average Standardized Ridge Coefficients"
)

axes[0].set_xlabel(
    "Average Coefficient"
)

axes[0].grid(
    axis="x",
    alpha=0.25
)

axes[1].barh(
    feature_plot_labels,
    feature_plot_df[
        "xgboost_average_importance"
    ].to_numpy(dtype=float),
    color="#ff7f0e"
)

axes[1].set_title(
    "Average XGBoost Feature Importance"
)

axes[1].set_xlabel(
    "Average Feature Importance"
)

axes[1].grid(
    axis="x",
    alpha=0.25
)

figure.suptitle(
    "Machine-Learning Feature Interpretation",
    fontsize=15,
    y=1.02
)

figure.tight_layout()

FEATURE_IMPORTANCE_FIGURE = (
    ML_FIGURE_DIR
    / "03_machine_learning_feature_interpretation.png"
)

figure.savefig(
    FEATURE_IMPORTANCE_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Machine-learning feature figure was saved.")

In [ ]:
ML_PREDICTION_FILE = (
    PROCESSED_DATA_DIR
    / "38_walk_forward_ml_predictions.parquet"
)

ML_TARGET_WEIGHT_FILE = (
    PROCESSED_DATA_DIR
    / "39_ml_target_weights.parquet"
)

ML_MONTHLY_RETURN_FILE = (
    PROCESSED_DATA_DIR
    / "40_ml_portfolio_monthly_returns.csv"
)

ML_PERFORMANCE_FILE = (
    PROCESSED_DATA_DIR
    / "41_ml_portfolio_performance.csv"
)

ML_ALPHA_FILE = (
    PROCESSED_DATA_DIR
    / "42_ml_factor_alpha.csv"
)

ML_MONTHLY_IC_FILE = (
    PROCESSED_DATA_DIR
    / "43_ml_monthly_information_coefficients.csv"
)

ML_IC_SUMMARY_FILE = (
    PROCESSED_DATA_DIR
    / "44_ml_information_coefficient_summary.csv"
)

ML_FEATURE_FILE = (
    PROCESSED_DATA_DIR
    / "45_ml_feature_summary.csv"
)

ML_MODEL_DIAGNOSTIC_FILE = (
    PROCESSED_DATA_DIR
    / "46_ml_annual_model_diagnostics.csv"
)

ml_prediction_df.to_parquet(
    ML_PREDICTION_FILE,
    index=False
)

ml_target_weights_df.to_parquet(
    ML_TARGET_WEIGHT_FILE,
    index=False
)

ml_strategy_backtest_df.to_csv(
    ML_MONTHLY_RETURN_FILE,
    index=False
)

ml_performance_summary_df.to_csv(
    ML_PERFORMANCE_FILE
)

ml_alpha_summary_df.to_csv(
    ML_ALPHA_FILE
)

monthly_prediction_ic_df.to_csv(
    ML_MONTHLY_IC_FILE,
    index=False
)

prediction_ic_summary_df.to_csv(
    ML_IC_SUMMARY_FILE,
    index=False
)

average_feature_summary_df.to_csv(
    ML_FEATURE_FILE
)

annual_model_diagnostic_df.to_csv(
    ML_MODEL_DIAGNOSTIC_FILE,
    index=False
)

print("Machine-learning result files were saved successfully.")

## Machine-Learning Conclusions

The machine-learning analysis used an annual expanding-window design
without random train-test splitting. Models were trained exclusively on
information available before each prediction period, producing 132
months of genuinely out-of-sample forecasts from 2015 through 2025.

Neither Ridge regression nor XGBoost improved return predictability.
Both models generated negative average out-of-sample Pearson and rank
information coefficients. In contrast, the standalone quality factor
produced a small positive information coefficient, although it was not
statistically significant.

The Ridge portfolio reduced volatility and maximum drawdown relative to
the equal-weighted benchmark, but its annualized return and information
ratio were lower. The XGBoost portfolio performed particularly poorly,
with high turnover, the deepest drawdown, and a statistically significant
negative factor-adjusted alpha.

Feature interpretation showed that quality received the highest average
XGBoost importance and a consistently positive Ridge coefficient.
However, historical feature importance did not translate into
out-of-sample predictive performance. This distinction demonstrates why
in-sample model interpretation should not be treated as evidence of
investable predictability.

Overall, model complexity did not improve the strategy. The transparent
quality-factor portfolio outperformed both machine-learning alternatives
after transaction costs. These findings emphasize the importance of
walk-forward validation, turnover control, and honest reporting of
negative out-of-sample results.